# CS164 Optimization Methods — Session 7 PCW: Gradient Descent

Reconstructed from the September 28, 2026 PCW screenshot. This notebook covers all four questions, the written responses, executable code, and regenerated results. The original Jupyter file was not provided.

## Question 1 — Line search review

**Exact line search:** Finds the step size minimizing the function along the chosen direction, generally at a greater computational cost.

**Approximate line search:** Finds a sufficiently good step satisfying acceptance conditions, even if it does not find the line minimum.

The **Wolfe conditions** consist of Armijo sufficient decrease, $\phi(\alpha)\leq\phi(0)+c_1\alpha\phi'(0)$, and curvature, $\phi'(\alpha)\geq c_2\phi'(0)$.

### Armijo backtracking
Starting with an initial step, while $f(x_i+\alpha d)>f(x_i)+c\alpha\nabla f(x_i)^Td$, reduce $\alpha\leftarrow p\alpha$ using $p=0.5$ and $c=10^{-4}$. Return the acceptable step.

*Technical note:* Armijo backtracking alone does not guarantee the Wolfe curvature condition.

In [1]:
# Backtracking: Armijo approximate line search
import numpy as np


def backtracking(f, grad_f, d, x_i, alpha=1.0, p=0.5, c=1e-4):
    """
    Armijo backtracking line search.

    Parameters
    ----------
    f : callable
        Objective function f(x).
    grad_f : callable
        Gradient of the objective function.
    d : np.ndarray
        Descent direction.
    x_i : np.ndarray
        Current position.
    alpha : float
        Initial step size (default: 1.0).
    p : float
        Step-size reduction factor (default: 0.5).
    c : float
        Armijo sufficient-decrease parameter (default: 1e-4).

    Returns
    -------
    float
        An acceptable step size satisfying the Armijo condition.
    """
    x_i = np.asarray(x_i, dtype=float)
    d = np.asarray(d, dtype=float)

    # Evaluate the function and gradient at x_i
    f_i = f(x_i)
    grad_i = grad_f(x_i)

    # Calculate the initial directional derivative
    slope = np.dot(grad_i, d)

    # The search direction must be a descent direction
    if slope >= 0:
        raise ValueError("d must be a descent direction.")

    # Reduce alpha until sufficient decrease is achieved
    while f(x_i + alpha * d) > f_i + c * alpha * slope:
        alpha = p * alpha

    # Return the accepted step size
    return alpha




### Backtracking tests

Quadratic bowl, shifted quadratic, and one-dimensional quadratic; in each case the expected step size is $\alpha=0.5$.

In [2]:
# Three tests for backtracking (as in the submitted PCW)
# Test 1: Quadratic bowl f(x,y) = x^2 + y^2
f1 = lambda x: x[0]**2 + x[1]**2
grad1 = lambda x: 2 * x
x1 = np.array([1.0, 1.0])
d1 = -grad1(x1)
alpha1 = backtracking(f1, grad1, d1, x1)
print("Test 1 - Quadratic bowl")
print(f"Step size: {alpha1}, New point: {x1 + alpha1 * d1}")
print()

# Test 2: Shifted quadratic f(x,y) = (x-3)^2 + (y+2)^2
f2 = lambda x: (x[0] - 3)**2 + (x[1] + 2)**2
grad2 = lambda x: np.array([2 * (x[0] - 3), 2 * (x[1] + 2)])
x2 = np.array([0.0, 0.0])
d2 = -grad2(x2)
alpha2 = backtracking(f2, grad2, d2, x2)
print("Test 2 - Shifted quadratic")
print(f"Step size: {alpha2}, New point: {x2 + alpha2 * d2}")
print()

# Test 3: One-dimensional quadratic f(x) = x^2
f3 = lambda x: x[0]**2
grad3 = lambda x: 2 * x
x3 = np.array([4.0])
d3 = -grad3(x3)
alpha3 = backtracking(f3, grad3, d3, x3)
print("Test 3 - One-dimensional quadratic")
print(f"Step size: {alpha3}, New point: {x3 + alpha3 * d3}")

assert np.isclose(alpha1, 0.5)
assert np.isclose(alpha2, 0.5)
assert np.isclose(alpha3, 0.5)
print("\nAll 3 tests passed!")




Test 1 - Quadratic bowl
Step size: 0.5, New point: [0. 0.]

Test 2 - Shifted quadratic
Step size: 0.5, New point: [ 3. -2.]

Test 3 - One-dimensional quadratic
Step size: 0.5, New point: [0.]

All 3 tests passed!


## Question 2 — Termination conditions

- **Absolute improvement:** stop if $f(x_i)-f(x_{i+1})<\epsilon$; the reduction has fallen below a fixed tolerance.
- **Relative improvement:** stop if $f(x_i)-f(x_{i+1})<\epsilon|f(x_i)|$; compares progress with the objective's current scale.
- **Gradient magnitude:** stop if $\|\nabla f(x_i)\|<\epsilon$; detects approximate stationarity.
- **Maximum iterations:** stop after a chosen iteration cap to avoid an endless computation.

Different combinations of termination conditions can better suit different objective scales and provide safeguards when the algorithm does not converge. A small gradient does not necessarily indicate a minimum.

## Question 3 — Gradient descent

Use the normalized negative gradient $d_i=-\nabla f(x_i)/\|\nabla f(x_i)\|$, choose $\alpha_i$ using backtracking, and update $x_{i+1}=x_i+\alpha_i d_i$. The implementation estimates the gradient with central finite differences, terminates when its norm is below $10^{-6}$, and caps iterations at 1000.

In [3]:
import numpy as np

def central_gradient(f, x, h=1e-5):
    grad = np.zeros_like(x, dtype=float)

    for j in range(len(x)):
        step = np.zeros_like(x, dtype=float)
        step[j] = h
        grad[j] = (f(x + step) - f(x - step)) / (2 * h)

    return grad


def gradient_descent(f, x0, epsilon=1e-6, max_iter=1000):
    """
    Minimize f using normalized gradient descent.

    Parameters
    ----------
    f : callable
        Objective function.
    x0 : np.ndarray
        Initial position.
    epsilon : float
        Gradient magnitude tolerance.
    max_iter : int
        Maximum number of iterations.

    Returns
    -------
    x : np.ndarray
        Final position.
    f(x) : float
        Objective function value at the final position.
    """
    x = np.array(x0, dtype=float, copy=True)

    # Gradient function used by both gradient descent and backtracking
    grad_f = lambda x: central_gradient(f, x)

    for i in range(max_iter):
        # Compute the gradient
        grad = grad_f(x)
        grad_norm = np.linalg.norm(grad)

        # Check the termination condition
        if grad_norm < epsilon:
            return x, f(x)

        # Calculate the normalized descent direction
        d = -grad / grad_norm

        # Find an acceptable step size using backtracking
        alpha = backtracking(f, grad_f, d, x)

        # Update the current position
        x_new = x + alpha * d

        # Check whether floating-point precision prevents progress
        if np.array_equal(x_new, x):
            raise RuntimeError("Step too small to make numerical progress.")

        x = x_new

    print("Maximum number of iterations reached.")
    return x, f(x)




## Question 4 — Quadratic bowl and saddle tests

For the bowl $f(x,y)=x^2+y^2$, the exact minimum is $(0,0)$ with objective value 0. Compare the computed position, distance and objective error.

In [4]:
# Test 1: Quadratic bowl and numerical-vs-exact comparison
def f(x):
    return x[0]**2 + x[1]**2

x0 = np.array([1.0, 1.0])
x_min, f_min = gradient_descent(f, x0)
true_min = np.array([0.0, 0.0])
true_f_min = 0.0

print("Test 1 - Quadratic Bowl:")
print("True minimum position:", true_min)
print("Computed position:", x_min)
print("Distance to true minimum:", np.linalg.norm(x_min - true_min))
print("True minimum value:", true_f_min)
print("Computed value:", f_min)
print("Objective error:", abs(f_min - true_f_min))




Test 1 - Quadratic Bowl:
True minimum position: [0. 0.]
Computed position: [2.69995362e-07 2.69995362e-07]
Distance to true minimum: 3.818311027646716e-07
True minimum value: 0.0
Computed value: 1.457949910384852e-13
Objective error: 1.457949910384852e-13


### Saddle point

For $f(x,y)=x^2-y^2$, starting from $(1,1)$, gradient descent decreases the objective until the 30-iteration limit is reached. The origin is a saddle point rather than a minimum; the function is unbounded below.

In [5]:
# Test 2: Saddle point f(x,y) = x^2 - y^2
def f(x):
    return x[0]**2 - x[1]**2

x0 = np.array([1.0, 1.0])

# Run gradient descent with a 30-iteration limit
x, f_final = gradient_descent(f, x0, max_iter=30)

print("\nTest 2 - Saddle Point")
print("Starting position:", x0)
print("Starting value:", f(x0))
print("Final position:", x)
print("Final value:", f_final)
print("Change in objective:", f_final - f(x0))

# The origin is a stationary saddle point, NOT a minimum
print("Saddle point:", np.array([0.0, 0.0]))
print("Function value at saddle:", f(np.array([0.0, 0.0])))




Maximum number of iterations reached.

Test 2 - Saddle Point
Starting position: [1. 1.]
Starting value: 0.0
Final position: [7.05919317e-03 3.06912846e+01]
Final value: -941.9549036159591
Change in objective: -941.9549036159591
Saddle point: [0. 0.]
Function value at saddle: 0.0


### Optional: exact line search on the saddle

Along the descent direction from $(1,1)$, the sliced objective is $\phi(\alpha)=-2\sqrt{2}\alpha$ and therefore has no finite minimizer. The function below detects this for the saddle objective.

In [6]:
# Optional: analytical exact line search for the saddle function

def exact_saddle_line_search(f, grad_f, d, x_i):
    """
    Analytical exact line search for f(x,y) = x^2 - y^2.
    Assumes d is a strict descent direction.
    """
    slope = np.dot(grad_f(x_i), d)
    curvature = d[0]**2 - d[1]**2

    if curvature <= 0:
        raise ValueError(
            "No finite optimal step: "
            "the function is unbounded below along this direction."
        )

    return -slope / (2 * curvature)




# Exact saddle-point line-search example
# Define the saddle function and its analytical gradient
def f_saddle(x):
    return x[0]**2 - x[1]**2


def grad_saddle(x):
    return np.array([2 * x[0], -2 * x[1]])


# Same starting point
x0 = np.array([1.0, 1.0])
grad = grad_saddle(x0)
d = -grad / np.linalg.norm(grad)

try:
    alpha = exact_saddle_line_search(f_saddle, grad_saddle, d, x0)
    x_new = x0 + alpha * d

    print("Exact step size:", alpha)
    print("New position:", x_new)
    print("New objective value:", f_saddle(x_new))
except ValueError as error:
    print(error)


No finite optimal step: the function is unbounded below along this direction.


**Class reminder:** Add arrows to contour plots during the in-class session.